In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import statsmodels.api as sm

PROCESSED_DATA_DIR = "data/processed"
ARTIFACTS_DIR      = os.path.join(PROCESSED_DATA_DIR, "model_artifacts")

with open(os.path.join(PROCESSED_DATA_DIR, "bin_edges.pkl"),     "rb") as f:
    bin_edges = pickle.load(f)
with open(os.path.join(PROCESSED_DATA_DIR, "woe_mappings.pkl"),  "rb") as f:
    woe_mappings = pickle.load(f)
with open(os.path.join(ARTIFACTS_DIR, "champion_scorecard.pkl"), "rb") as f:
    pd_model = pickle.load(f)

df_oot = pd.read_csv(
    os.path.join(PROCESSED_DATA_DIR, "01_oot_cohort.csv.gz"),
    compression='gzip', low_memory=False
)
print(f"OOT cohort loaded: {df_oot.shape}  |  default rate: {df_oot['target'].mean():.2%}")

# ── Numeric columns: bin using saved cut-points ───────────────────────────────
for col, edges in bin_edges.items():
    bin_col = f"bin_{col}"
    woe_col = f"{bin_col}_WoE"
    if bin_col not in woe_mappings or col not in df_oot.columns:
        continue
    df_oot[bin_col] = pd.cut(df_oot[col], bins=edges, include_lowest=True).astype(str)
    df_oot[woe_col] = df_oot[bin_col].map(woe_mappings[bin_col]).fillna(0.0)

# ── Categorical columns: bin label IS the raw string value ───────────────────
for bin_col, mapping in woe_mappings.items():
    col     = bin_col.replace("bin_", "")
    woe_col = f"{bin_col}_WoE"
    if col in bin_edges:          # already handled above
        continue
    if col not in df_oot.columns:
        continue
    if woe_col in df_oot.columns: # already populated
        continue
    df_oot[bin_col] = df_oot[col].astype(str)
    df_oot[woe_col] = df_oot[bin_col].map(mapping).fillna(0.0)

# ── Score: align column order exactly to training ─────────────────────────────
# NOTE: the champion was saved with statsmodels remove_data() (slim pickle), which
# nulls model.exog_names — so we recover the feature order from params.index instead.
model_woe_cols = [c for c in pd_model.params.index if c != "const"]

# Safety: if any training column is absent from OOT (e.g. a category never seen),
missing = [c for c in model_woe_cols if c not in df_oot.columns]
if missing:
    print(f"⚠️  {len(missing)} WoE column(s) absent in OOT — filling with 0.0: {missing}")
for c in missing:
    df_oot[c] = 0.0

X_oot = sm.add_constant(df_oot[model_woe_cols], has_constant="add")
df_oot["computed_PD"] = pd_model.predict(X_oot)

print(f"OOT scoring complete. Mean predicted PD: {df_oot['computed_PD'].mean():.4f}")

OOT cohort loaded: (225639, 51)  |  default rate: 21.29%


OOT scoring complete. Mean predicted PD: 0.1906


In [2]:
def psi_status(val):
    if val < 0.10: return "STABLE"
    if val < 0.25: return "MINOR SHIFT"
    return "CRITICAL DRIFT"


def calculate_psi_continuous(expected, actual, num_bins=10):
    """PSI for a CONTINUOUS score (e.g. predicted PD). Bins are decile cut-points of
    the expected (development) distribution. < 0.1 stable | 0.1–0.25 monitor | > 0.25 retrain."""
    bins = np.percentile(expected, np.linspace(0, 100, num_bins + 1))
    bins[0], bins[-1] = -np.inf, np.inf
    bins = np.unique(bins)                      # guard against duplicate edges
    expected_pct = np.histogram(expected, bins=bins)[0] / len(expected)
    actual_pct   = np.histogram(actual,   bins=bins)[0] / len(actual)
    expected_pct = np.where(expected_pct == 0, 1e-4, expected_pct)
    actual_pct   = np.where(actual_pct   == 0, 1e-4, actual_pct)
    return np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))


def calculate_psi_discrete(expected_labels, actual_labels):
    """PSI for a DISCRETE/binned feature. Compares the population SHARE that falls in
    each WoE bin between dev and OOT. This is the correct treatment for scorecard
    features — they take only a handful of distinct bin values, so percentile-binning
    their WoE column (the previous approach) collapses to degenerate, meaningless PSI."""
    e = expected_labels.value_counts(normalize=True)
    a = actual_labels.value_counts(normalize=True)
    cats = e.index.union(a.index)
    e = e.reindex(cats, fill_value=0.0).clip(lower=1e-4)
    a = a.reindex(cats, fill_value=0.0).clip(lower=1e-4)
    return float(np.sum((a - e) * np.log(a / e)))


# ── Output-score PSI: dev PD distribution vs OOT PD distribution ──────────────
df_dev_scored = pd.read_csv(
    os.path.join(PROCESSED_DATA_DIR, "03_scored_champion_output.csv.gz"),
    compression='gzip'
)
score_psi = calculate_psi_continuous(df_dev_scored['computed_PD'].values,
                                     df_oot['computed_PD'].values)

print("=== PORTFOLIO DRIFT MONITORING (PSI) ===\n")
print(f"Output Score PSI : {score_psi:.4f}  →  {psi_status(score_psi)}\n")

# ── Per-feature PSI: compare bin-membership shares (dev vs OOT) ────────────────
df_dev_eng = pd.read_csv(
    os.path.join(PROCESSED_DATA_DIR, "02_engineered_features.csv.gz"),
    compression='gzip', low_memory=False
)

print(f"{'Feature':<40} {'PSI':>8}  Status")
print("-" * 70)

critical = []
feature_psi_rows = []
for bin_col in woe_mappings.keys():                 # only model-retained features
    if bin_col not in df_dev_eng.columns or bin_col not in df_oot.columns:
        continue
    psi_val = calculate_psi_discrete(df_dev_eng[bin_col].astype(str),
                                     df_oot[bin_col].astype(str))
    status  = psi_status(psi_val)
    feature_psi_rows.append({'Feature': bin_col, 'PSI': round(psi_val, 4), 'Status': status})
    print(f"{bin_col:<40} {psi_val:>8.4f}  {status}")
    if psi_val >= 0.25:
        critical.append(bin_col)

# Persist the monitoring table for the model-validation report / dashboards
psi_report = pd.DataFrame(
    [{'Metric': 'output_score_PSI', 'PSI': round(score_psi, 4), 'Status': psi_status(score_psi)}]
    + feature_psi_rows
)
psi_report.to_csv(os.path.join(ARTIFACTS_DIR, "psi_monitoring_report.csv"), index=False)

if critical:
    print(f"\n{len(critical)} feature(s) with critical drift (PSI >= 0.25): {critical}")
else:
    print("\nNo individual features show critical drift.")
print(f"\nSaved monitoring table → model_artifacts/psi_monitoring_report.csv")

=== PORTFOLIO DRIFT MONITORING (PSI) ===

Output Score PSI : 0.0059  →  STABLE



Feature                                       PSI  Status
----------------------------------------------------------------------
bin_loan_amnt                              0.0180  STABLE
bin_term                                   0.0003  STABLE
bin_int_rate                               0.0138  STABLE
bin_grade                                  0.0105  STABLE
bin_sub_grade                              0.0317  STABLE
bin_home_ownership                         0.0073  STABLE
bin_annual_inc                             0.0054  STABLE


bin_verification_status                    0.0232  STABLE
bin_dti                                    0.0038  STABLE
bin_fico_range_low                         0.0349  STABLE
bin_tot_cur_bal                            0.0293  STABLE
bin_total_rev_hi_lim                       0.0369  STABLE
bin_acc_open_past_24mths                   0.0243  STABLE
bin_avg_cur_bal                            0.0288  STABLE


bin_bc_open_to_buy                         0.1038  MINOR SHIFT
bin_bc_util                                0.1535  MINOR SHIFT
bin_mo_sin_old_rev_tl_op                   0.0333  STABLE
bin_mo_sin_rcnt_rev_tl_op                  0.0174  STABLE
bin_mo_sin_rcnt_tl                         0.0076  STABLE
bin_mort_acc                               0.0091  STABLE
bin_num_actv_rev_tl                        0.0472  STABLE

No individual features show critical drift.

Saved monitoring table → model_artifacts/psi_monitoring_report.csv
